# Article -> Country (tf-idf calculation)
Here we calculate the incidence rates (document frequency) of all the countries weighted by how many pagelinks point to them on Wikipedia. We can then use these frequencies for adjusting the proportion of countries that are linked to in a given Wikipedia article. This effectively requires more evidence of linking for countries like the USA/UK/France and less evidence for much "smaller" countries like Ecuador or Greenland.

In [1]:
import wmfdata

In [2]:
spark = wmfdata.spark.create_session(app_name='pyspark reg; regions; isaacj',
                                  type='yarn-regular', # local, yarn-regular, yarn-large
                                  )  

SPARK_HOME: /usr/lib/spark3
Using Hadoop client lib jars at 3.2.0, provided by Spark.
PYSPARK_PYTHON=/opt/conda-analytics/bin/python3


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/04/30 17:13:55 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in mesos/standalone/kubernetes and LOCAL_DIRS in YARN).
24/04/30 17:13:56 WARN Utils: Service 'sparkDriver' could not bind on port 12000. Attempting port 12001.
24/04/30 17:13:56 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
24/04/30 17:14:04 WARN Utils: Service 'org.apache.spark.network.netty.NettyBlockTransferService' could not bind on port 13000. Attempting port 13001.
24/04/30 17:14:04 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Attempted to request executors before the AM has registered!


In [3]:
wikidata_snapshot = '2024-04-01'
tablename = 'isaacj.qid_to_country'

In [6]:
"""
Base incidence rate -- weighted by link incidence
NOTE: using some old data that was precalculated but
I expect it shouldn't change much over time.
"""

query = f"""
SELECT
  COALESCE(country, "") AS country,
  COUNT(1) AS num_links
FROM isaacj.outlinks_allwikis l
LEFT JOIN {tablename} c
  ON (l.qid_to = c.qid)
WHERE
  l.snapshot = '2023-01'
  AND c.snapshot = '{wikidata_snapshot}'
GROUP BY
  COALESCE(country, "")
ORDER BY
  num_links DESC
"""

print(query)
spark.sql(query).show(300, False)


SELECT
  COALESCE(country, "") AS country,
  COUNT(1) AS num_links
FROM isaacj.outlinks_allwikis l
LEFT JOIN isaacj.qid_to_country c
  ON (l.qid_to = c.qid)
WHERE
  l.snapshot = '2023-01'
  AND c.snapshot = '2024-04-01'
GROUP BY
  COALESCE(country, "")
ORDER BY
  num_links DESC



24/04/30 17:20:18 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.


+---------------------------------------------+---------+
|country                                      |num_links|
+---------------------------------------------+---------+
|France                                       |653508011|
|United States                                |467035978|
|Italy                                        |330842753|
|Mexico                                       |197337734|
|Russia                                       |180398671|
|United Kingdom                               |140974152|
|Germany                                      |123088745|
|Spain                                        |114690737|
|Japan                                        |99493950 |
|Iran                                         |83150801 |
|India                                        |73685770 |
|Ukraine                                      |61740219 |
|China                                        |60443739 |
|Poland                                       |59566468 |
|Brazil       

In [7]:
"""
Base incidence rate -- weighted by link incidence
This is the denominator for the numerators in the previous cell
"""

query = f"""
SELECT
  COUNT(1) AS num_links
FROM isaacj.outlinks_allwikis
WHERE
  snapshot = '2023-01'
"""

print(query)
spark.sql(query).show(300, False)


SELECT
  COUNT(1) AS num_links
FROM isaacj.outlinks_allwikis
WHERE
  snapshot = '2023-01'



+----------+
|num_links |
+----------+
|3522932131|
+----------+



In [9]:
"""
Base incidence rate -- weighted by link incidence
This is how many non-geographic links there are.
"""

query = f"""
SELECT
  COUNT(1) AS num_links
FROM isaacj.outlinks_allwikis l
LEFT ANTI JOIN {tablename} c
  ON (l.qid_to = c.qid AND c.snapshot = '{wikidata_snapshot}')
WHERE
  l.snapshot = '2023-01'
"""

print(query)
spark.sql(query).show(300, False)


SELECT
  COUNT(1) AS num_links
FROM isaacj.outlinks_allwikis l
LEFT ANTI JOIN isaacj.qid_to_country c
  ON (l.qid_to = c.qid AND c.snapshot = '2024-04-01')
WHERE
  l.snapshot = '2023-01'



+----------+
|num_links |
+----------+
|1320188104|
+----------+



## Calculating IDF values
I did this in a spreadsheet separately but the general formula is:
* `Document frequency = <number of links to country> / <total number of links>`
* `IDF = log10(1 / document-frequency)`

So for France, it would look like:
```
df: 653,508,011 / 3,522,932,131 = 0.186
idf: log10(1 / 0.186) = 0.73
```


## Alternative approaches

In [6]:
"""
Base incidence rate -- equally-weighted articles
NOTE: I didn't use this but leaving for documentation purposes
"""

query = f"""
WITH relevant_wikis AS (
    SELECT
      DISTINCT(database_code) AS wiki_db
    FROM canonical_data.wikis
    WHERE
      database_group = 'wikipedia'
      AND status = 'open'
      AND visibility = 'public'
      AND editability = 'public'
),
overall AS (
    SELECT
      COUNT(DISTINCT(item_id)) AS num_items
    FROM wmf.wikidata_item_page_link wd
    INNER JOIN relevant_wikis db
      ON (wd.wiki_db = db.wiki_db)
    WHERE
      snapshot = '{wikidata_snapshot}'
      AND page_namespace = 0
),
groundtruth AS (
    SELECT DISTINCT
      qid,
      country
    FROM {tablename}
    WHERE
      snapshot = '{wikidata_snapshot}'
),
overall_country_counts AS (
   SELECT
      country,
      COUNT(1) AS num_items
    FROM groundtruth
    GROUP BY
      country
)
SELECT
  country,
  c.num_items AS num_qids,
  ROUND(c.num_items / o.num_items, 8) AS term_freq
FROM overall_country_counts c
CROSS JOIN overall o
ORDER BY
  term_freq DESC
"""

print(query)
spark.sql(query).show(300, False)


WITH relevant_wikis AS (
    SELECT
      DISTINCT(database_code) AS wiki_db
    FROM canonical_data.wikis
    WHERE
      database_group = 'wikipedia'
      AND status = 'open'
      AND visibility = 'public'
      AND editability = 'public'
),
overall AS (
    SELECT
      COUNT(DISTINCT(item_id)) AS num_items
    FROM wmf.wikidata_item_page_link wd
    INNER JOIN relevant_wikis db
      ON (wd.wiki_db = db.wiki_db)
    WHERE
      snapshot = '2024-04-01'
      AND page_namespace = 0
),
groundtruth AS (
    SELECT DISTINCT
      qid,
      country
    FROM isaacj.qid_to_country
    WHERE
      snapshot = '2024-04-01'
),
overall_country_counts AS (
   SELECT
      country,
      COUNT(1) AS num_items
    FROM groundtruth
    GROUP BY
      country
)
SELECT
  country,
  c.num_items AS num_qids,
  ROUND(c.num_items / o.num_items, 8) AS term_freq
FROM overall_country_counts c
CROSS JOIN overall o
ORDER BY
  term_freq DESC



24/04/30 16:57:04 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.


+---------------------------------------------+--------+----------+
|country                                      |num_qids|term_freq |
+---------------------------------------------+--------+----------+
|United States                                |1787968 |0.08084151|
|Germany                                      |629206  |0.02844903|
|Japan                                        |489558  |0.02213496|
|France                                       |486498  |0.02199661|
|Russia                                       |466512  |0.02109296|
|United Kingdom                               |435024  |0.01966925|
|Norway                                       |433766  |0.01961237|
|Canada                                       |396692  |0.0179361 |
|Mexico                                       |372437  |0.01683943|
|Spain                                        |371801  |0.01681068|
|India                                        |357697  |0.01617298|
|Italy                                        |3